In [9]:

const fs = await import("node:fs");

const years = Array.from(
  { length: 12 },
  (_, i) => String(2016 + i)
);

// Read the CSV file
const csv = fs.readFileSync(
  "UniversityRankings.csv",
  "utf8"
);

// Parse CSV, including quoted fields and commas
function parseCSV(text) {
  const rows = [];
  let row = [];
  let field = "";
  let quoted = false;

  text = text.replace(/^\uFEFF/, "");

  for (let i = 0; i < text.length; i++) {
    const c = text[i];

    if (c === '"') {
      if (quoted && text[i + 1] === '"') {
        field += '"';
        i++;
      } else {
        quoted = !quoted;
      }
    } else if (c === "," && !quoted) {
      row.push(field);
      field = "";
    } else if ((c === "\n" || c === "\r") && !quoted) {
      if (c === "\r" && text[i + 1] === "\n") i++;
      row.push(field);
      if (row.some(v => v.trim() !== "")) rows.push(row);
      row = [];
      field = "";
    } else {
      field += c;
    }
  }

  row.push(field);
  if (row.some(v => v.trim() !== "")) rows.push(row);

  const headers = rows.shift().map(h => h.trim());

  return rows.map(values =>
    Object.fromEntries(
      headers.map((h, i) => [h, values[i] ?? ""])
    )
  );
}

const data = parseCSV(csv);

// Convert rankings to numbers
const rankings = data.map(row => {
  const university = {
    name: row["University Name"],
    ipeds: row["IPEDS"]
  };

  years.forEach(year => {
    const value = String(row[year] ?? "").trim();
    const number = value === "" ? NaN : Number(value);

    university[year] = Number.isFinite(number)
      ? number
      : null;
  });

  return university;
});

// Select universities ranked 50 or better in 2016
const top50 = rankings
  .filter(row =>
    row["2016"] !== null &&
    row["2016"] <= 50
  )
  .sort((a, b) => a["2016"] - b["2016"]);

const universities = top50.map(row => row.name);

// Ranking values for blue cells
const z = top50.map(row =>
  years.map(year => {
    const value = row[year];
    return value !== null && value <= 50
      ? value
      : null;
  })
);

// All displayed labels
const labels = top50.map(row =>
  years.map(year => {
    const value = row[year];

    if (value === null) return "";
    if (value > 50) return ">50";

    return String(Math.trunc(value));
  })
);

// Gray background for every cell
const background = {
  z: top50.map(() => years.map(() => 1)),
  x: years,
  y: universities,
  type: "heatmap",
  colorscale: [
    [0, "lightgrey"],
    [1, "lightgrey"]
  ],
  showscale: false,
  hoverinfo: "skip",
  xgap: 1,
  ygap: 1
};

// Blue cells for ranks 1 through 50

const heatmap = {
  z: z.map(row => row.map(v => v === null ? null : 51 - v)),
  x: years,
  y: universities,
  type: "heatmap",

  colorscale: [
    [0.00, "#F7FBFF"],
    [0.20, "#DEEBF7"],
    [0.40, "#BDD7E7"],
    [0.60, "#6BAED6"],
    [0.80, "#2171B5"],
    [1.00, "#08306B"]
  ],

  zmin: 1,
  zmax: 50,

  colorbar: {
    title: "Rank",
    tickvals: [1, 11, 21, 31, 41, 50],
    ticktext: ["50", "40", "30", "20", "10", "1"]
  },

  xgap: 1,
  ygap: 1
};

// Display >50 on gray cells
const annotations = [];

top50.forEach((row, i) => {
  years.forEach(year => {
    const value = row[year];

    let label = "";
    let fontColor = "black";

    if (value === null) {
      label = "";
    } else if (value > 50) {
      label = ">50";
      fontColor = "black";
    } else {
      label = String(Math.trunc(value));

      // White text for the two darkest blue ranges
      if (value <= 20) {
        fontColor = "white";
      } else {
        fontColor = "black";
      }
    }

    if (label !== "") {
      annotations.push({
        x: year,
        y: universities[i],
        text: label,
        showarrow: false,
        font: {
          size: 10,
          color: fontColor
        }
      });
    }
  });
});

// Chart layout
const layout = {
  title: "University Rankings (2016–2027)",
  width: 1100,
  height: Math.max(800, universities.length * 28),
  margin: {
    l: 290,
    r: 100,
    t: 70,
    b: 80
  },
  
xaxis: {
  title: "Year",
  tickmode: "array",
  tickvals: years,
  ticktext: years,
  tickangle: -45,
  side: "bottom"
},
  yaxis: {
    title: "University",
    autorange: "reversed",
    type: "category",
    categoryorder: "array",
    categoryarray: universities
  },
  annotations: annotations,
  plot_bgcolor: "lightgrey"
};

// Generate an HTML visualization from JavaScript
const html = `
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>University Rankings Heatmap</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
</head>
<body>
<div id="heatmap"></div>
<script>
const background = ${JSON.stringify(background)};
const heatmap = ${JSON.stringify(heatmap)};
const layout = ${JSON.stringify(layout)};

Plotly.newPlot(
  "heatmap",
  [background, heatmap],
  layout
);
</script>
</body>
</html>
`;

fs.writeFileSync(
  "UniversityRankingsHeatmap.html",
  html
);

console.log("Heatmap created successfully!");
console.log("Universities included:", top50.length);
console.log("Open UniversityRankingsHeatmap.html in Chrome.");



Heatmap created successfully!
Universities included: 50
Open UniversityRankingsHeatmap.html in Chrome.
